# NamedVertexCoarsener example

This notebook contracts explicitly selected connected components. It compares selection by label and by UID, and compares contracting every component with contracting only the largest component.


In [ ]:
from tree_coarsening import NamedVertexCoarsener
from tree_coarsening.utils import make_named_component_tree

G = make_named_component_tree(
    component_sizes=(5, 3),
    selected_labels=("A", "B"),
    include_singleton=True,
    seed=0,
)
print(f"raw tree: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

for node in G.nodes:
    data = G.nodes[node]
    parent = next(iter(G.predecessors(node)), None)
    parent_uid = None if parent is None else G.nodes[parent]["uid"]
    print(
        f"uid={data['uid']!r} | label={data['label']!r} | "
        f"parent_uid={parent_uid!r}"
    )


In [ ]:
def show_encoded(title, graph):
    print(f"\n{title}: {graph.number_of_nodes()} nodes")
    for node, data in graph.nodes(data=True):
        print(f"Node {node}: {data}")

by_label_all = NamedVertexCoarsener(
    labels={"A", "B"},
    component_policy="all",
).fit([G])
H_label_all = by_label_all.transform(G)
show_encoded("labels A/B, all components", H_label_all)


In [ ]:
by_label_largest = NamedVertexCoarsener(
    labels={"A", "B"},
    component_policy="largest",
).fit([G])
H_label_largest = by_label_largest.transform(G)
show_encoded("labels A/B, largest component", H_label_largest)


In [ ]:
selected_uids = {
    data["uid"]
    for _, data in G.nodes(data=True)
    if data["uid"].startswith("named_component_0_")
}

by_uid = NamedVertexCoarsener(
    uids=selected_uids,
    component_policy="all",
).fit([G])
H_uid = by_uid.transform(G)
show_encoded("first component selected by UID", H_uid)


In [ ]:
print("UID-selected vocabulary entries:")
for token in by_uid.encoder_.vocab.creation_order:
    entry = by_uid.encoder_.vocab[token]
    print(f"{token}: P={entry.parent}, L={entry.label}, A={entry.attach}")


In [ ]:
def uid_signature(graph):
    nodes = sorted(
        (data["uid"], data["label"], data["time"])
        for _, data in graph.nodes(data=True)
    )
    edges = sorted(
        (graph.nodes[parent]["uid"], graph.nodes[child]["uid"])
        for parent, child in graph.edges
    )
    return nodes, edges

for coarsener, encoded in (
    (by_label_all, H_label_all),
    (by_label_largest, H_label_largest),
    (by_uid, H_uid),
):
    assert uid_signature(coarsener.decode(encoded)) == uid_signature(G)

print("all roundtrips ok")
